<a href="https://colab.research.google.com/github/Riz2693/Higo-Technical-Test/blob/main/1.%20Data%20Dummy%20Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install library dan import module**

In [18]:
!pip install faker
!pip install scipy

In [19]:
import pandas as pd
import numpy as np
import random
import scipy.stats as st
from faker import Faker
from datetime import datetime

# **Inisialisasi faker dan mengatur seed untuk mengatur randomness**

In [20]:
# Inisialisasi Faker dengan data lokal Indonesia
fake = Faker('id_ID')

# Mengatur seed agar data acak tetap konsisten jika dijalankan ulang
Faker.seed(42)
random.seed(42)
np.random.seed(42)

# **Pembuatan data dummy dan tambahan fitur**

In [21]:
# Daftar referensi untuk data acak
lokasi_list = ['Kopi Kenangan Senayan', 'Starbucks Kemang', 'Plaza Indonesia', 'Perpustakaan Nasional', 'Stasiun Sudirman']
tipe_lokasi_list = ['Cafe', 'Cafe', 'Mall', 'Fasilitas Umum', 'Transportasi']
merk_hp_list = ['Samsung', 'Apple', 'Xiaomi', 'Oppo', 'Vivo']
kategori_minat_digital = ['F&B & Culinary Lifestyle', 'F&B & Culinary Lifestyle', 'E-Book & Digital Education', 'Entertainment & Streaming', 'Travel & Leisure']

data = []
jumlah_data = 1000 # Jumlah baris data yang ingin dibuat

for _ in range(jumlah_data):
    # Memilih lokasi dan tipe lokasi yang saling berpasangan
    idx_lokasi = random.randint(0, len(lokasi_list) - 1)

    # Membuat jam login acak (format HH:MM:SS)
    jam_login = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"

    # Menambahkan satu baris data (dictionary) ke dalam list
    data.append({
        'Nama Lokasi': lokasi_list[idx_lokasi],
        'Tipe Lokasi': tipe_lokasi_list[idx_lokasi],
        'Jam Login': jam_login,
        'Nama': fake.name(),
        'Email': fake.free_email(),
        'No Telepon': fake.phone_number(),
        'Tahun Lahir': random.randint(1980, 2008), # Asumsi pengguna usia 18-46 tahun
        'Merk HP': random.choice(merk_hp_list),
        'Minat Digital': random.choice(kategori_minat_digital),
        'CTR': round(random.uniform(0, 15.0), 1), # Skor minat antara 0 hingga 15
        'Durasi Akses (Menit/Minggu)': random.randint(60, 600) # Skor waktu 60 menit hingga 600 menit
    })

In [22]:
# Mengubah list menjadi tabel Pandas DataFrame
df = pd.DataFrame(data)
df.head(5)

,Nama Lokasi,Tipe Lokasi,Jam Login,Nama,Email,No Telepon,Tahun Lahir,Merk HP,Minat Digital,CTR,Durasi Akses (Menit/Minggu)
0,Kopi Kenangan Senayan,Cafe,00:47:17,"Balidin Dongoran, S.T.",ivanmandala@hotmail.com,+62 (081) 960-0133,1987,Apple,F&B & Culinary Lifestyle,11.0,149
1,Stasiun Sudirman,Transportasi,13:02:01,Drs. Akarsana Purwanti,salsabilapalastri@yahoo.com,+62-0794-026-5423,1982,Apple,F&B & Culinary Lifestyle,7.6,87
2,Stasiun Sudirman,Transportasi,06:45:41,"Tgk. Baktiono Iswahyudi, S.Sos",tambamalik@hotmail.com,+62 (0078) 161 8495,2002,Vivo,Entertainment & Streaming,3.3,344
3,Kopi Kenangan Senayan,Cafe,05:44:27,R.A. Tiara Hidayat,wijayantijaga@yahoo.com,+62 (031) 647 5255,1990,Xiaomi,F&B & Culinary Lifestyle,3.2,404
4,Kopi Kenangan Senayan,Cafe,02:24:06,Mumpuni Santoso,sudiatirahmi@gmail.com,0832764835,1991,Xiaomi,Travel & Leisure,4.0,104


# **Ekstraksi Fitur**

In [23]:
# Menentukan Batas Minimum & Maksimum Logis (Bisa disesuaikan dengan target platform)
# Untuk Durasi, asumsi maks logis adalah 600 menit. Untuk CTR, maks industri rata-rata 15%
min_durasi, max_durasi = 0, 600
min_ctr, max_ctr = 0.0, 15.0

# Proses Normalisasi ke Skala 0 - 100
df['Skor CTR'] = ((df['CTR'] - min_ctr) / (max_ctr - min_ctr)) * 100
df['Skor Durasi Akses (Menit/Minggu)'] = ((df['Durasi Akses (Menit/Minggu)'] - min_durasi) / (max_durasi - min_durasi)) * 100

# Menghitung Skor Minat Digital Komposit (Bobot 50% Durasi : 50% CTR)
df['Skor Minat Digital'] = (df['Skor Durasi Akses (Menit/Minggu)'] * 0.5) + (df['Skor CTR'] * 0.5)
df['Skor Minat Digital'] = df['Skor Minat Digital'].round(1)

In [24]:
# Mengubah ke Level Numerik ke Kategorikal untuk Kemudahan Analisis
bins = [0, 40, 70, 100]
labels = ['Rendah', 'Sedang', 'Tinggi']

df['Level Respons CTR'] = pd.cut(df['Skor CTR'], bins=bins, labels=labels, include_lowest=True)
df['Level Durasi Akses'] = pd.cut(df['Skor Durasi Akses (Menit/Minggu)'], bins=bins, labels=labels, include_lowest=True)
df['Level Minat Digital'] = pd.cut(df['Skor Minat Digital'], bins=bins, labels=labels, include_lowest=True)

In [25]:
# Membuat variabel 'Usia' dari 'Tahun Lahir'
tahun_sekarang = datetime.now().year
df['Usia'] = tahun_sekarang - df['Tahun Lahir']

In [26]:
# Membuat variabel 'Sesi Login' dari 'Jam Login'
def kategori_waktu(jam_str):
    jam = int(jam_str.split(':')[0])
    if 5 <= jam < 12: return 'Pagi'
    elif 12 <= jam < 15: return 'Siang'
    elif 15 <= jam < 18: return 'Sore'
    else: return 'Malam'

df['Sesi Login'] = df['Jam Login'].apply(kategori_waktu)

In [27]:
# Menampilkan hasil
df.head(5)

,Nama Lokasi,Tipe Lokasi,Jam Login,Nama,Email,No Telepon,Tahun Lahir,Merk HP,Minat Digital,CTR,Durasi Akses (Menit/Minggu),Skor CTR,Skor Durasi Akses (Menit/Minggu),Skor Minat Digital,Level Respons CTR,Level Durasi Akses,Level Minat Digital,Usia,Sesi Login
0,Kopi Kenangan Senayan,Cafe,00:47:17,"Balidin Dongoran, S.T.",ivanmandala@hotmail.com,+62 (081) 960-0133,1987,Apple,F&B & Culinary Lifestyle,11.0,149,73.333333,24.833333,49.1,Tinggi,Rendah,Sedang,39,Malam
1,Stasiun Sudirman,Transportasi,13:02:01,Drs. Akarsana Purwanti,salsabilapalastri@yahoo.com,+62-0794-026-5423,1982,Apple,F&B & Culinary Lifestyle,7.6,87,50.666667,14.500000,32.6,Sedang,Rendah,Rendah,44,Siang
2,Stasiun Sudirman,Transportasi,06:45:41,"Tgk. Baktiono Iswahyudi, S.Sos",tambamalik@hotmail.com,+62 (0078) 161 8495,2002,Vivo,Entertainment & Streaming,3.3,344,22.000000,57.333333,39.7,Rendah,Sedang,Rendah,24,Pagi
3,Kopi Kenangan Senayan,Cafe,05:44:27,R.A. Tiara Hidayat,wijayantijaga@yahoo.com,+62 (031) 647 5255,1990,Xiaomi,F&B & Culinary Lifestyle,3.2,404,21.333333,67.333333,44.3,Rendah,Sedang,Sedang,36,Pagi
4,Kopi Kenangan Senayan,Cafe,02:24:06,Mumpuni Santoso,sudiatirahmi@gmail.com,0832764835,1991,Xiaomi,Travel & Leisure,4.0,104,26.666667,17.333333,22.0,Rendah,Rendah,Rendah,35,Malam


# **Validasi data dengan interval kepercayaan**

In [29]:
# Hitung Interval Kepercayaan 95% untuk 'Rata-rata Usia'
data_usia = df['Usia']

# Hitung rata-rata sampel
mean_usia = np.mean(data_usia)

# Hitung Standard Error of the Mean (SEM)
sem_usia = st.sem(data_usia)

# Hitung batas bawah dan batas atas Interval Kepercayaan (95%)
derajat_kebebasan = len(data_usia) - 1
ci_bawah, ci_atas = st.t.interval(confidence=0.95, df=derajat_kebebasan, loc=mean_usia, scale=sem_usia)

In [30]:
print("--- Hasil Analisis Interval Kepercayaan (95%) ---")
print(f"Berdasarkan sampel {jumlah_data} pengguna:")
print(f"Rata-rata Usia            : {mean_usia:.2f} tahun")
print(f"Interval Kepercayaan (CI) : Antara {ci_bawah:.2f} hingga {ci_atas:.2f} tahun")
print("Interpretasi: Kita yakin 95% bahwa rata-rata usia seluruh populasi pengguna berada di rentang tersebut.")

--- Hasil Analisis Interval Kepercayaan (95%) ---
Berdasarkan sampel 1000 pengguna:
Rata-rata Usia            : 31.51 tahun
Interval Kepercayaan (CI) : Antara 30.98 hingga 32.04 tahun
Interpretasi: Kita yakin 95% bahwa rata-rata usia seluruh populasi pengguna berada di rentang tersebut.


In [31]:
# Hitung Interval Kepercayaan 95% untuk 'Rata-rata Skor Minat Digital'
data_minat = df['Skor Minat Digital']

# Hitung rata-rata sampel
mean_minat = np.mean(data_minat)

# Hitung Standard Error of the Mean (SEM)
sem_minat = st.sem(data_minat)

# Hitung batas bawah dan batas atas Interval Kepercayaan (95%)
derajat_kebebasan = len(data_minat) - 1
ci_bawah, ci_atas = st.t.interval(confidence=0.95, df=derajat_kebebasan, loc=mean_minat, scale=sem_minat)

In [32]:
print("--- Hasil Analisis Interval Kepercayaan (95%) ---")
print(f"Berdasarkan sampel {jumlah_data} pengguna:")
print(f"Rata-rata Skor Minat Digital : {mean_minat:.2f}")
print(f"Interval Kepercayaan (CI)    : Antara {ci_bawah:.2f} hingga {ci_atas:.2f}")
print("Interpretasi: Kita yakin 95% bahwa rata-rata populasi pengguna memiliki ketertarikan digital di level Sedang.")

--- Hasil Analisis Interval Kepercayaan (95%) ---
Berdasarkan sampel 1000 pengguna:
Rata-rata Skor Minat Digital : 52.41
Interval Kepercayaan (CI)    : Antara 51.22 hingga 53.60
Interpretasi: Kita yakin 95% bahwa rata-rata populasi pengguna memiliki ketertarikan digital di level Sedang.


# **Konversi ke csv untuk analisis lebih lanjut**

In [33]:
df.to_csv('data_dummy_pengguna.csv', index=False)